# Final experiments — Knowledge-driven agent on a real mountain road
**Case study:** Mexican Federal Highway 54, Villanueva – Apozol (Zacatecas), three most curved 5-km segments.

**Agent:** curvature-aware speed planning from the road map, sight-distance rule, dense/soft obstacle handling with
overtaking through the opposite lane under oncoming traffic, and imperfect perception (limited sight distance on curves,
range noise, and classification error).

**Design:** 3 segments × 4 classification-error levels × 2 decision policies × 30 seeds (+ no-agent baseline),
100 km/h cruise speed, 300 veh/h oncoming traffic. Results include 95 % Wilson confidence intervals.

**Before starting:** upload **`fed54_villanueva_apozol.zip`** (from v4a.1) to the Colab file panel (root).
Then *Runtime → Run all*. Expected duration: about 45–60 min.

The experiment saves a checkpoint after every run. If Colab disconnects, run all cells again: completed runs are skipped.
Set `USE_DRIVE = True` to keep the checkpoint in Google Drive (recommended).

## 1. Setup

In [ ]:
!pip install -q eclipse-sumo==1.27.1 traci==1.27.1 sumolib==1.27.1 pyproj rtree

import os, sumo
os.environ["SUMO_HOME"] = sumo.SUMO_HOME
os.environ["PATH"] = os.path.join(sumo.SUMO_HOME, "bin") + os.pathsep + os.environ["PATH"]
!sumo --version | head -1

USE_DRIVE = False          # True = store the checkpoint in Google Drive
OUT = "final_results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/agent_fed54_final"
os.makedirs(OUT, exist_ok=True)
print("Results folder:", OUT)

## 2. Modules

In [ ]:
%%writefile ruta_osm.py
"""
Importación de una carretera real desde OpenStreetMap y análisis de su curvatura.
Caso de estudio: Carretera Federal 54, tramo Villanueva – Apozol (Zacatecas).
"""
import math, os, subprocess, time
import numpy as np
import requests
import sumolib

# netconvert necesita la base de proyecciones (proj.db) para convertir lat/lon a metros
if os.environ.get("SUMO_HOME"):
    _proj = os.path.join(os.environ["SUMO_HOME"], "data", "proj")
    if os.path.isdir(_proj):
        os.environ.setdefault("PROJ_DATA", _proj)
        os.environ.setdefault("PROJ_LIB", _proj)

UA = {"User-Agent": "fed54-curve-agent/1.0 (academic research)"}
OVERPASS = ["https://overpass-api.de/api/interpreter",
            "https://overpass.private.coffee/api/interpreter",
            "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.kumi.systems/api/interpreter"]

def _overpass(consulta, valida, rondas=3, espera=30):
    """Envía la consulta a varios servidores Overpass, con reintentos. Devuelve la respuesta válida."""
    errores = []
    for ronda in range(rondas):
        for url in OVERPASS:
            try:
                r = requests.post(url, data={"data": consulta}, headers=UA, timeout=240)
                if r.ok and valida(r):
                    return r
                errores.append(f"{url.split('/')[2]}: HTTP {r.status_code}")
            except requests.RequestException as e:
                errores.append(f"{url.split('/')[2]}: {type(e).__name__}")
        if ronda < rondas - 1:
            print(f"Servidores ocupados; reintento en {espera} s...")
            time.sleep(espera)
    raise RuntimeError("No se pudo consultar Overpass. Detalle: " + "; ".join(errores[-len(OVERPASS):]) +
                       ". Intenta en unos minutos o sube el archivo .osm de una corrida anterior.")

# ---------------- 1. Geocodificación ----------------
def geocodificar(lugar):
    """Devuelve (lat, lon) de un lugar usando Nominatim (OpenStreetMap)."""
    r = requests.get("https://nominatim.openstreetmap.org/search",
                     params={"q": lugar, "format": "json", "limit": 1}, headers=UA, timeout=60)
    r.raise_for_status()
    d = r.json()
    if not d:
        raise ValueError(f"No se encontró: {lugar}")
    time.sleep(1)                      # política de uso de Nominatim: 1 consulta por segundo
    return float(d[0]["lat"]), float(d[0]["lon"])

# ---------------- 2. Descarga de la carretera ----------------
def descargar_carretera(origen, destino, ref_regex, archivo, margen=0.08):
    """Descarga solo las vías con la referencia indicada (p. ej. la Federal 54)
    dentro del rectángulo que contiene origen y destino, con un margen en grados."""
    s = min(origen[0], destino[0]) - margen; n = max(origen[0], destino[0]) + margen
    w = min(origen[1], destino[1]) - margen; e = max(origen[1], destino[1]) + margen
    q = f"""[out:xml][timeout:180];
(
  way["highway"]["ref"~"{ref_regex}"]({s},{w},{n},{e});
);
(._;>;);
out body;"""
    if os.path.exists(archivo) and os.path.getsize(archivo) > 1000:
        print(f"Usando el archivo ya existente: {archivo}")
        return archivo
    r = _overpass(q, lambda r: b"<way" in r.content)
    open(archivo, "wb").write(r.content)
    return archivo

# ---------------- 3. Conversión a red de SUMO ----------------
def convertir_a_sumo(archivo_osm, archivo_net):
    if not os.path.exists(archivo_osm):
        raise FileNotFoundError(f"No existe el archivo {archivo_osm}")
    r = subprocess.run(["netconvert", "--osm-files", archivo_osm, "-o", archivo_net,
                        "--geometry.remove", "false",        # conservar la geometría de las curvas
                        "--keep-edges.components", "1",      # solo el componente conectado principal
                        "--no-turnarounds", "true",
                        "--osm.sidewalks", "false",
                        "--tls.discard-simple", "true",
                        "--opposites.guess", "true",          # carril contrario para rebasar
                        "--no-warnings", "true"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("netconvert falló:\n" + (r.stderr or r.stdout)[-2000:])
    return sumolib.net.readNet(archivo_net, withInternal=False)

# ---------------- 4. Ruta entre origen y destino ----------------
def _aristas_cercanas(net, lat, lon, radio=500):
    x, y = net.convertLonLat2XY(lon, lat)
    cand = net.getNeighboringEdges(x, y, radio)
    cand.sort(key=lambda t: t[1])
    return [e for e, _ in cand if e.allows("passenger")]

def construir_ruta(net, origen, destino):
    """Ruta más corta (lista de aristas) de origen a destino sobre la red importada."""
    mejor = None
    for radio in (300, 800, 2000, 5000):
        for e1 in _aristas_cercanas(net, *origen, radio)[:6]:
            for e2 in _aristas_cercanas(net, *destino, radio)[:6]:
                ruta, costo = net.getShortestPath(e1, e2)
                if ruta and (mejor is None or costo < mejor[1]):
                    mejor = (ruta, costo)
        if mejor:
            return mejor[0]
    raise RuntimeError("No se encontró una ruta conectada entre origen y destino.")

# ---------------- 5. Perfil de curvatura ----------------
def geometria_ruta(ruta):
    """Polilínea (x, y) de la ruta (carril 0 de cada arista) y límite de velocidad (km/h) en cada punto."""
    pts, vel = [], []
    for e in ruta:
        for p in e.getLane(0).getShape():
            if not pts or math.dist(p, pts[-1]) > 0.5:
                pts.append(p); vel.append(e.getSpeed() * 3.6)
    return np.array(pts), np.array(vel)

def remuestrear(pts, paso=5.0, vel=None):
    seg = np.hypot(*np.diff(pts, axis=0).T)
    s = np.concatenate([[0], np.cumsum(seg)])
    s_new = np.arange(0, s[-1], paso)
    P = np.column_stack([np.interp(s_new, s, pts[:, 0]), np.interp(s_new, s, pts[:, 1])])
    if vel is None:
        return s_new, P
    idx = np.clip(np.searchsorted(s, s_new, side="right") - 1, 0, len(vel) - 1)
    return s_new, P, vel[idx]

def perfil_curvatura(ruta, paso=5.0, semiventana=20.0, suavizado=7):
    """Curvatura con signo (1/m) cada `paso` metros: círculo por tres puntos separados
    `semiventana` m, suavizado con media móvil. Positiva = curva a la izquierda.
    Devuelve (s, P, kap, vel): distancia, puntos (x, y), curvatura y límite de velocidad (km/h)."""
    pts, vel = geometria_ruta(ruta)
    s, P, v = remuestrear(pts, paso, vel)
    k = max(1, int(semiventana / paso))
    kap = np.zeros(len(P))
    for i in range(k, len(P) - k):
        a, b, c = P[i - k], P[i], P[i + k]
        ab, bc, ca = np.linalg.norm(b - a), np.linalg.norm(c - b), np.linalg.norm(a - c)
        cruz = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
        if ab * bc * ca > 0:
            kap[i] = 2 * cruz / (ab * bc * ca)
    kap = np.convolve(kap, np.ones(suavizado) / suavizado, mode="same")
    return s, P, kap, v

def detectar_curvas(s, kap, radio_max=400.0, long_min=20.0):
    """Tramos continuos con radio < radio_max. Devuelve lista de dicts."""
    en_curva = np.abs(kap) > 1.0 / radio_max
    curvas, i = [], 0
    while i < len(s):
        if en_curva[i]:
            j = i
            while j + 1 < len(s) and en_curva[j + 1] and np.sign(kap[j + 1]) == np.sign(kap[i]):
                j += 1
            if s[j] - s[i] >= long_min:
                kmax = np.max(np.abs(kap[i:j + 1]))
                curvas.append({"inicio_m": round(s[i]), "fin_m": round(s[j]),
                               "longitud_m": round(s[j] - s[i]), "radio_min_m": round(1 / kmax),
                               "sentido": "izquierda" if kap[i] > 0 else "derecha"})
            i = j + 1
        else:
            i += 1
    return curvas

def mejores_tramos(curvas, long_total, ventana=5000.0, n=3, radio_critico=250.0,
                   s=None, urbano=None, max_frac_urbana=0.05):
    """Ventanas de `ventana` m con más curvas cerradas (radio mínimo < radio_critico), sin traslape.
    Si se dan `s` y `urbano`, descarta ventanas con más de `max_frac_urbana` de zona urbana."""
    criticas = [c for c in curvas if c["radio_min_m"] < radio_critico]
    candidatos = []
    for c in criticas:
        ini = max(0.0, c["inicio_m"] - 200.0)
        fin = min(long_total, ini + ventana)
        if fin - ini < 0.8 * ventana:
            continue                                  # ventana recortada al final de la ruta
        if urbano is not None:
            m = (s >= ini) & (s <= fin)
            if urbano[m].mean() > max_frac_urbana:
                continue
        dentro = [d for d in criticas if d["inicio_m"] >= ini and d["fin_m"] <= fin]
        candidatos.append((len(dentro), -min(d["radio_min_m"] for d in dentro), ini, fin))
    candidatos.sort(reverse=True)
    elegidos = []
    for cnt, _, ini, fin in candidatos:
        if all(fin <= a or ini >= b for a, b, _ in elegidos):
            elegidos.append((ini, fin, cnt))
        if len(elegidos) == n:
            break
    return [{"inicio_km": round(a / 1000, 2), "fin_km": round(b / 1000, 2), "curvas_cerradas": c}
            for a, b, c in sorted(elegidos)]


# ---------------- 6. Exclusión de zonas urbanas ----------------
RADIO_LOCALIDAD = {"city": 2500.0, "town": 1500.0, "village": 700.0, "hamlet": 300.0}

def descargar_localidades(origen, destino, margen=0.08):
    """Ciudades, pueblos y villas de OpenStreetMap dentro del rectángulo de la ruta."""
    s = min(origen[0], destino[0]) - margen; n = max(origen[0], destino[0]) + margen
    w = min(origen[1], destino[1]) - margen; e = max(origen[1], destino[1]) + margen
    q = f"""[out:json][timeout:120];
node["place"~"^(city|town|village|hamlet)$"]({s},{w},{n},{e});
out;"""
    def valida(r):
        try:
            return "elements" in r.json()
        except ValueError:
            return False
    r = _overpass(q, valida)
    return [{"nombre": el["tags"].get("name", "?"), "tipo": el["tags"]["place"],
             "lat": el["lat"], "lon": el["lon"]} for el in r.json()["elements"]]

def localidades_manuales(lista):
    """Localidades agregadas a mano. Cada elemento: (texto, tipo, radio_m o None) o
    (texto, tipo, radio_m o None, (lat, lon)) si ya conoces sus coordenadas.
    Si una localidad no se encuentra, se avisa y se omite (no detiene el notebook)."""
    res = []
    for item in lista:
        texto, tipo, radio = item[:3]
        coord = item[3] if len(item) > 3 else None
        nombre = texto.split(",")[0].strip()
        if coord is None:
            partes = [p.strip() for p in texto.split(",")]
            variantes = [texto, f"{nombre}, Zacatecas, México", f"{nombre}, Zacatecas", nombre]
            for v in dict.fromkeys(variantes):
                try:
                    coord = geocodificar(v)
                    break
                except ValueError:
                    continue
        if coord is None:
            print(f"AVISO: no se encontró «{texto}». Agrégala con sus coordenadas: "
                  f'("{nombre}", "{tipo}", None, (lat, lon))')
            continue
        res.append({"nombre": nombre, "tipo": tipo, "lat": coord[0], "lon": coord[1],
                    "radio": radio, "manual": True})
    return res

def marcar_urbano(net, s, P, vel, localidades, vel_urbana=60.0, radios=RADIO_LOCALIDAD, alcance_aviso=4000.0):
    """True en los puntos de la ruta dentro de una localidad o con límite <= vel_urbana km/h.
    Devuelve (mascara, cruzadas, cercanas): `cercanas` son localidades a menos de `alcance_aviso` m
    de la ruta que NO se marcaron (útil para revisar si falta alguna)."""
    urbano = vel <= vel_urbana + 1e-6
    cruzadas, cercanas, vistos = [], [], set()
    for loc in localidades:
        x, y = net.convertLonLat2XY(loc["lon"], loc["lat"])
        d = np.hypot(P[:, 0] - x, P[:, 1] - y)
        r = loc.get("radio") or radios.get(loc["tipo"], 700.0)
        i = int(np.argmin(d))
        fila = {"localidad": loc["nombre"], "tipo": loc["tipo"], "km": round(float(s[i]) / 1000, 1),
                "distancia_m": round(float(d[i])), "radio_m": round(r),
                "origen": "manual" if loc.get("manual") else "OSM"}
        clave = (loc["nombre"].lower(), fila["km"])
        if d[i] < r:
            urbano |= d < r
            if clave not in vistos:
                cruzadas.append(fila); vistos.add(clave)
        elif d[i] < alcance_aviso and loc["tipo"] != "hamlet":
            cercanas.append(fila)
    return urbano, sorted(cruzadas, key=lambda c: c["km"]), sorted(cercanas, key=lambda c: c["km"])

In [ ]:
%%writefile agente_ruta.py
"""
v4c — Agente knowledge-driven (con incertidumbre de percepción) sobre una carretera real (Federal 54, Villanueva–Apozol).
- Conocimiento del mapa: perfil de curvatura de la ruta (radio en cada punto).
- Planeación de velocidad anticipada: nunca exceder la velocidad segura de las curvas próximas.
- Obstáculo denso: el agente proyecta un "obstáculo virtual" sobre el carril y decide
  cuándo rebasar por el carril contrario, según el tráfico de frente.
- Obstáculo blando: se ignora.
"""
import math, os, csv, json, random
import numpy as np
import traci, sumolib
import ruta_osm as ro

# ---------------- Parámetros del agente ----------------
A_LAT_MAX        = 3.0     # m/s²  aceleración lateral máxima admisible
DECEL_CONFORT    = 2.0     # m/s²  desaceleración para la planeación de velocidad
MARGEN_CURVA     = 10.0    # m     llegar a la velocidad segura antes de la curva
HORIZONTE        = 400.0   # m     distancia de anticipación sobre el mapa
RANGO_DETECCION  = 200.0   # m     alcance de detección de obstáculos (sensor de largo alcance)
V_REBASE         = 8.0     # m/s   velocidad máxima para rebasar por el carril contrario
DIST_VISIBLE_MIN = 150.0   # m     tramo de carril contrario que debe estar libre
T_MARGEN_REBASE  = 4.0     # s     margen de tiempo adicional para el rebase
# Percepción con incertidumbre (v4c)
DESPEJE_LATERAL  = 5.0     # m     distancia del carril al obstáculo visual (talud, cerro) en curvas
SIGMA_REL        = 0.03    # —     ruido de la distancia medida: σ = SIGMA_REL·d + SIGMA_ABS
SIGMA_ABS        = 0.5     # m
DIST_CONFIRMACION = 50.0   # m     a esta distancia se vuelve a clasificar el obstáculo (segunda mirada)
DECEL_EMERG      = 4.0     # m/s²  desaceleración para detenerse dentro de la distancia visible
REGLA_VISIBILIDAD = True   # nunca circular más rápido de lo que permite detenerse en la distancia visible
CALENTAMIENTO    = 500.0   # m     el vehículo entra antes del tramo; las métricas solo cuentan dentro del tramo
L_EGO, ANCHO_OBST = 4.5, 3.0
EGO = "ego"

# ---------------- Mapa (base de conocimiento geométrica) ----------------
class Mapa:
    def __init__(self, net, aristas, paso=5.0):
        self.net = net
        self.aristas = [e.getID() for e in aristas]
        self.cum, acc = {}, 0.0
        for e in aristas:
            self.cum[e.getID()] = acc
            acc += e.getLane(0).getLength()
        self.longitud = acc
        s, P, kap, vel = ro.perfil_curvatura(aristas, paso=paso)
        # reescalar para que coincida con las posiciones de SUMO (longitudes de carril)
        self.s = s * (self.longitud / max(s[-1], 1.0))
        self.kap = kap
        R = 1.0 / np.maximum(np.abs(kap), 1e-4)
        self.R = R
        self.vseg = np.sqrt(A_LAT_MAX * R)
        self.paso = self.s[1] - self.s[0]
        # arista contraria (mismo par de nodos, sentido opuesto)
        self.contraria = {}
        for e in aristas:
            for o in e.getToNode().getOutgoing():
                if o.getToNode() == e.getFromNode():
                    self.contraria[o.getID()] = e.getID()

    def s_de(self, edge, pos):
        return self.cum[edge] + pos if edge in self.cum else None

    def radio(self, s):
        i = int(np.clip(round(s / self.paso), 0, len(self.R) - 1))
        return float(self.R[i])

    def v_permitida(self, s, v_lim):
        """Máxima velocidad actual que permite llegar a cada curva próxima a su velocidad segura."""
        i0 = int(np.clip(s // self.paso, 0, len(self.s) - 1))
        i1 = int(np.clip((s + HORIZONTE) // self.paso, i0 + 1, len(self.s)))
        d = np.maximum(self.s[i0:i1] - s - MARGEN_CURVA, 0.0)
        v = np.sqrt(self.vseg[i0:i1] ** 2 + 2 * DECEL_CONFORT * d)
        return float(min(v_lim, v.min()))

    def distancia_visible(self, s, hasta=None):
        """Distancia de visibilidad en curva (despeje lateral M): S = 2R·acos(1 − M/R),
        con el radio mínimo entre s y s+hasta. En recta queda limitada por el alcance del sensor."""
        hasta = RANGO_DETECCION if hasta is None else max(hasta, self.paso)
        i0 = int(np.clip(s // self.paso, 0, len(self.R) - 1))
        i1 = int(np.clip((s + hasta) // self.paso + 1, i0 + 1, len(self.R)))
        R = float(self.R[i0:i1].min())
        if R <= DESPEJE_LATERAL:
            return DESPEJE_LATERAL
        return float(min(RANGO_DETECCION, 2 * R * math.acos(1 - DESPEJE_LATERAL / R)))

    def s_contrario(self, edge, pos):
        """Posición equivalente sobre la ruta de un vehículo que circula en sentido contrario."""
        e = self.contraria.get(edge)
        if e is None:
            return None
        return self.cum[e] + self.net.getEdge(e).getLane(0).getLength() - pos

# ---------------- Escenario de un tramo ----------------
def preparar_tramo(carpeta, mapa, ini_m, fin_m, vel_kmh, obstaculos, flujo_vph, t_salida=60.0):
    """obstaculos: lista de (id, s_m, densidad). Escribe rutas y POIs del tramo."""
    os.makedirs(carpeta, exist_ok=True)
    def ubicar(s):
        for e in reversed(mapa.aristas):
            if mapa.cum[e] <= s:
                return e, s - mapa.cum[e]
        return mapa.aristas[0], s
    e0, p0 = ubicar(max(0.0, ini_m - CALENTAMIENTO)); e1, p1 = ubicar(fin_m)
    ruta = mapa.aristas[mapa.aristas.index(e0): mapa.aristas.index(e1) + 1]
    contraria = [k for e in reversed(ruta) for k, v in mapa.contraria.items() if v == e]
    v = vel_kmh / 3.6
    with open(f"{carpeta}/tramo.rou.xml", "w") as f:
        f.write(f'''<routes>
    <vType id="auto_autonomo" accel="2.6" decel="4.5" emergencyDecel="9.0" sigma="0" length="{L_EGO}"
           maxSpeed="{v:.2f}" speedFactor="2" guiShape="passenger" color="0,0.6,1"/>
    <vType id="frente" accel="2.6" decel="4.5" sigma="0.5" length="4.5" maxSpeed="25" speedFactor="1"/>
    <vType id="virtual" length="{ANCHO_OBST}" maxSpeed="0.1" color="1,0,0" guiShape="passenger"/>
    <route id="r_ego" edges="{' '.join(ruta)}"/>
''')
        if flujo_vph > 0 and contraria:
            f.write(f'    <route id="r_frente" edges="{" ".join(contraria)}"/>\n')
            f.write(f'    <flow id="frente" type="frente" route="r_frente" begin="0" end="7200" '
                    f'vehsPerHour="{flujo_vph}" departSpeed="max" departLane="best"/>\n')
        f.write(f'    <vehicle id="{EGO}" type="auto_autonomo" route="r_ego" depart="{t_salida}" '
                f'departPos="{p0:.1f}" arrivalPos="{min(p1, mapa.net.getEdge(e1).getLength()-1):.1f}" '
                f'departSpeed="{v:.2f}"/>\n</routes>\n')
    colores = {"denso": "1,0,0", "blando": "0,1,0"}
    info = []
    with open(f"{carpeta}/obstaculos.add.xml", "w") as f:
        f.write("<additional>\n")
        for oid, s, dens in obstaculos:
            e, p = ubicar(s)
            info.append({"id": oid, "s": s, "edge": e, "pos": p, "densidad": dens})
            f.write(f'    <poi id="{oid}" lane="{e}_0" pos="{p:.1f}" color="{colores[dens]}" type="{dens}" '
                    f'width="{ANCHO_OBST}" height="{ANCHO_OBST}"><param key="densidad" value="{dens}"/></poi>\n')
        f.write("</additional>\n")
    tramo = {"ruta": ruta, "obstaculos": info, "ini_m": ini_m, "fin_m": fin_m, "vel_kmh": vel_kmh,
             "flujo_vph": flujo_vph, "archivo_net": getattr(mapa, "archivo_net", "")}
    with open(f"{carpeta}/tramo.json", "w") as f:
        json.dump(tramo, f, indent=2)
    return tramo

def obstaculos_aleatorios(mapa, curvas, ini_m, fin_m, rng, n_densos=2, separacion=600.0):
    """Coloca obstáculos densos al azar en curvas cerradas del tramo, y uno blando 150 m después de cada uno."""
    cand = [c for c in curvas if ini_m + 300 < c["inicio_m"] and c["fin_m"] < fin_m - 300
            and c["radio_min_m"] < 250]
    rng.shuffle(cand)
    elegidos = []
    for c in cand:
        s = (c["inicio_m"] + c["fin_m"]) / 2
        if all(abs(s - x) > separacion for x in elegidos):
            elegidos.append(s)
        if len(elegidos) == n_densos:
            break
    obs = []
    for k, s in enumerate(sorted(elegidos)):
        obs += [(f"denso_{k}", s, "denso"), (f"blando_{k}", min(s + 150, fin_m - 50), "blando")]
    return obs

# ---------------- Agente ----------------
def frente_libre(mapa, s_ego, v_ego, d_obs):
    """True si no viene tráfico de frente en la distancia necesaria para rebasar."""
    t_rebase = (d_obs + ANCHO_OBST + L_EGO + 20.0) / max(V_REBASE, 1.0) + T_MARGEN_REBASE
    necesaria = max(DIST_VISIBLE_MIN, d_obs + 25.0 * t_rebase)   # 25 m/s: velocidad máx. de frente
    for v in traci.vehicle.getIDList():
        if v == EGO or v.startswith("virtual"):
            continue
        s = mapa.s_contrario(traci.vehicle.getRoadID(v), traci.vehicle.getLanePosition(v))
        if s is not None and 0 < s - s_ego < necesaria:
            return False
    # vehículos que ya van por el carril contrario frente al ego también cuentan
    return True

def _clasificar(real, p_error, rng):
    """Clasificación con error: con probabilidad p_error se confunde denso con blando y viceversa."""
    if rng.random() < p_error:
        return "blando" if real == "denso" else "denso"
    return real

def correr(carpeta, mapa, tramo, con_agente=True, gui=False, seed=42, delay=100, guardar_pasos=False,
           p_error=0.0, politica="conservadora"):
    """politica: 'conservadora' = si alguna mirada dice "denso", se trata como denso (seguridad primero);
                 'ultima'       = se actúa según la clasificación más reciente."""
    try:
        traci.close()
    except Exception:
        pass
    cmd = ["sumo-gui" if gui else "sumo", "-n", mapa.archivo_net, "-r", f"{carpeta}/tramo.rou.xml",
           "-a", f"{carpeta}/obstaculos.add.xml", "--seed", str(seed), "--step-length", "0.1",
           "--no-step-log", "--no-warnings", "--collision.action", "warn", "--lanechange.duration", "2",
           "--end", "3000"]
    if gui:
        cmd += ["--start", "--quit-on-end", "--delay", str(delay)]
    traci.start(cmd)
    dens = {o["id"]: o for o in tramo["obstaculos"]}
    virtuales, pasos = {}, []
    rng = random.Random(seed * 7919 + int(p_error * 1000))
    creencia, mirada_cercana, virtual_blando = {}, set(), set()
    deteccion_d, info_virtual = {}, {}
    colision_obst, colision_veh, rebases, t_espera = set(), 0, 0, 0.0
    s_prev, salio, siguiendo = None, False, False
    if con_agente:
        pass
    while traci.simulation.getMinExpectedNumber() > 0:
        traci.simulationStep()
        t = traci.simulation.getTime()
        ids = traci.vehicle.getIDList()
        # fijar los obstáculos virtuales recién insertados
        for oid, (vid, frente) in list(virtuales.items()):
            if vid in ids and not traci.vehicle.isStopped(vid):
                try:
                    traci.vehicle.setStop(vid, dens[oid]["edge"], pos=frente, laneIndex=0, duration=1e6)
                except traci.TraCIException:
                    pass
        if EGO not in ids:
            if salio:
                break
            continue
        if not salio:
            salio = True
            if con_agente:
                traci.vehicle.setParameter(EGO, "laneChangeModel.lcOpposite", "0")
            if gui:
                traci.gui.trackVehicle("View #0", EGO); traci.gui.setZoom("View #0", 1500)
        edge = traci.vehicle.getRoadID(EGO)
        lane = traci.vehicle.getLaneID(EGO)
        pos = traci.vehicle.getLanePosition(EGO)
        v = traci.vehicle.getSpeed(EGO)
        s = mapa.s_de(edge, pos)
        en_contrario = edge in mapa.contraria
        if en_contrario:
            s = mapa.s_contrario(edge, pos)
        if s is None:                       # dentro de un cruce interno
            s = (s_prev or 0.0) + v * 0.1
        s_prev = s
        R = mapa.radio(s)
        a_lat = v * v / R
        # ---- percepción: visibilidad en curva, clasificación con error y ruido en la distancia
        d_obs, obs_id, d_est = math.inf, None, math.inf
        for oid, o in dens.items():
            d = o["s"] - s
            if d <= 0 or d > RANGO_DETECCION:
                continue
            if not con_agente:
                continue
            if oid not in creencia:
                if d <= mapa.distancia_visible(s, d):          # ¿ya es visible desde aquí?
                    creencia[oid] = _clasificar(o["densidad"], p_error, rng)
                    deteccion_d[oid] = d
                else:
                    continue
            if d <= DIST_CONFIRMACION and oid not in mirada_cercana:
                nueva = _clasificar(o["densidad"], p_error, rng)          # segunda mirada, más cerca
                if politica == "conservadora" and creencia[oid] == "denso":
                    nueva = "denso"                                       # nunca se degrada un "denso"
                creencia[oid] = nueva
                mirada_cercana.add(oid)
            if d < d_obs:
                d_obs, obs_id = d, oid
        if obs_id is not None:
            sigma = SIGMA_REL * d_obs + SIGMA_ABS
            d_est = max(0.5, d_obs + rng.gauss(0.0, sigma))
        accion = "sin_agente"
        if con_agente:
            v_lim = traci.vehicle.getAllowedSpeed(EGO)
            v_obj = mapa.v_permitida(s, max(v_lim, 0.1))
            accion = "reducir_velocidad" if v_obj < v_lim - 0.5 else "mantener_trayectoria"
            if REGLA_VISIBILIDAD:
                v_vis = math.sqrt(2 * DECEL_EMERG * max(mapa.distancia_visible(s) - 15.0, 0.0))
                if v_vis < v_obj:
                    v_obj, accion = v_vis, "reducir_por_visibilidad"
            cre = creencia.get(obs_id)
            # el agente cambió de opinión: era "denso" y ahora cree "blando" → retira el obstáculo virtual
            if obs_id in virtuales and cre == "blando" and not en_contrario:
                vid, _ = virtuales.pop(obs_id)
                if vid in ids:
                    traci.vehicle.remove(vid)
            if obs_id is not None and cre == "denso":
                o = dens[obs_id]
                if obs_id not in virtuales:
                    # obstáculo virtual en la posición ESTIMADA, alargado según la incertidumbre (±2σ)
                    sigma = SIGMA_REL * d_est + SIGMA_ABS
                    largo = ANCHO_OBST + 4 * sigma
                    centro = o["pos"] + (d_est - d_obs)
                    L_edge = mapa.net.getEdge(o["edge"]).getLength()
                    frente = float(np.clip(centro + largo / 2, largo, L_edge - 0.1))
                    vid = f"virtual_{obs_id}"
                    traci.vehicle.add(vid, "r_ego", typeID="virtual", departPos=f"{frente:.1f}",
                                      departSpeed="0", departLane="0")
                    traci.vehicle.setRoute(vid, [o["edge"]])
                    traci.vehicle.setLength(vid, largo)
                    virtuales[obs_id] = (vid, frente)
                    info_virtual[obs_id] = {"s_frente": mapa.cum[o["edge"]] + frente, "largo": largo}
                    if o["densidad"] == "blando":
                        virtual_blando.add(obs_id)
                vid, frente = virtuales[obs_id]
                iv = info_virtual[obs_id]
                if vid not in ids:
                    # aún no está en el carril: detenerse antes de la parte trasera del obstáculo estimado
                    s_trasera = iv["s_frente"] - iv["largo"]
                    v_obj = min(v_obj, math.sqrt(max(0.0, 2 * DECEL_EMERG * (s_trasera - s - 8.0))))
                    if v < 0.5 and vid in traci.simulation.getPendingVehicles():
                        # detenido y el virtual no cabe: recortarlo para que quede delante del vehículo
                        traci.vehicle.remove(vid)
                        nuevo = iv["s_frente"] - (s + 4.0)
                        if nuevo >= ANCHO_OBST:
                            vid2 = f"{vid}_r{int(t*10)}"
                            traci.vehicle.add(vid2, "r_ego", typeID="virtual", departPos=f"{frente:.1f}",
                                              departSpeed="0", departLane="0")
                            traci.vehicle.setRoute(vid2, [o["edge"]])
                            traci.vehicle.setLength(vid2, nuevo)
                            virtuales[obs_id] = (vid2, frente); iv["largo"] = nuevo
                        else:
                            virtuales.pop(obs_id)          # demasiado tarde: el obstáculo está encima
                            creencia[obs_id] = "ignorado"
                if frente_libre(mapa, s, v, d_est):
                    traci.vehicle.setParameter(EGO, "laneChangeModel.lcOpposite", "1")
                    v_obj = min(v_obj, V_REBASE) if d_est < 60 else v_obj
                    accion = "rebasar_contrario"
                else:
                    traci.vehicle.setParameter(EGO, "laneChangeModel.lcOpposite", "0")
                    accion = "esperar_frente_libre"
            elif not en_contrario:
                traci.vehicle.setParameter(EGO, "laneChangeModel.lcOpposite", "0")
            traci.vehicle.setSpeed(EGO, v_obj)
        # métricas
        for oid, o in dens.items():
            if o["densidad"] == "denso" and not en_contrario and lane == f"{o['edge']}_0" \
               and abs((pos - L_EGO / 2) - o["pos"]) < L_EGO / 2 + ANCHO_OBST / 2:
                colision_obst.add(oid)
        if EGO in traci.simulation.getCollidingVehiclesIDList():
            colision_veh += 1
        if v < 1.0:
            t_espera += 0.1
        pasos.append({"t": t, "s_m": round(s, 1), "edge": edge, "contrario": int(en_contrario),
                      "v_kmh": round(v * 3.6, 1), "radio_m": round(R, 1), "a_lat": round(a_lat, 2),
                      "d_obs": round(d_obs, 1) if math.isfinite(d_obs) else None,
                      "d_est": round(d_est, 1) if math.isfinite(d_est) else None,
                      "obs": dens[obs_id]["densidad"] if obs_id else "ninguno",
                      "creencia": creencia.get(obs_id, "ninguno") if obs_id else "ninguno",
                      "accion": accion})
    traci.close()
    if guardar_pasos and pasos:
        nombre = "pasos_agente.csv" if con_agente else "pasos_base.csv"
        with open(f"{carpeta}/{nombre}", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=pasos[0].keys()); w.writeheader(); w.writerows(pasos)
    # métricas solo dentro del tramo (se excluye la zona de calentamiento)
    pasos_m = [p for p in pasos if tramo["ini_m"] <= p["s_m"] <= tramo["fin_m"]] or pasos
    a = np.array([p["a_lat"] for p in pasos_m]) if pasos_m else np.zeros(1)
    t_espera = sum(0.1 for p in pasos_m if p["v_kmh"] < 3.6)
    cambios = [p["contrario"] for p in pasos]
    rebases = sum(1 for i in range(1, len(cambios)) if cambios[i] == 1 and cambios[i - 1] == 0)
    n_densos = sum(1 for o in dens.values() if o["densidad"] == "denso")
    n_blandos = sum(1 for o in dens.values() if o["densidad"] == "blando")
    det = [deteccion_d[k] for k in deteccion_d if dens[k]["densidad"] == "denso"]
    return {"condicion": "con_agente" if con_agente else "sin_agente", "p_error": p_error,
            "politica": politica if con_agente else "—",
            "obst_densos": n_densos, "obst_blandos": n_blandos,
            "colision_obst_denso": len(colision_obst), "colision_vehiculo": int(colision_veh > 0),
            "maniobra_innecesaria": len(virtual_blando),
            "dist_deteccion_m": round(float(np.mean(det)), 1) if det else None,
            "rebases": rebases, "a_lat_max": round(float(a.max()), 2),
            "t_sobre_umbral_s": round(float((a > A_LAT_MAX + 0.05).sum() * 0.1), 1),
            "t_espera_s": round(t_espera, 1),
            "tiempo_recorrido_s": round(pasos_m[-1]["t"] - pasos_m[0]["t"], 1) if pasos_m else None,
            "v_media_kmh": round(float(np.mean([p["v_kmh"] for p in pasos_m])), 1) if pasos_m else None}

In [ ]:
%%writefile ver_tramo.py
"""
Visualiza en tu PC (sumo-gui) un tramo real de la Federal 54 con el agente.
Coloca este archivo en la carpeta descomprimida del escenario y ejecuta:
    python ver_tramo.py
Requisitos (una sola vez):  pip install numpy requests sumolib==1.27.1 traci==1.27.1
"""
import json, os
os.chdir(os.path.dirname(os.path.abspath(__file__)))
import sumolib
import agente_ruta as ag

CON_AGENTE = True            # False = caso base (sin agente)
P_ERROR    = 0.0             # probabilidad de clasificar mal un obstáculo (0.0 – 1.0)
POLITICA   = "conservadora"  # "conservadora" o "ultima"
SEMILLA    = 1
DELAY_MS   = 50

if __name__ == "__main__":
    cfg = json.load(open("tramo.json"))
    net = sumolib.net.readNet(cfg["archivo_net"], withInternal=False)
    ruta = [net.getEdge(e) for e in open("ruta_aristas.txt").read().split()]
    mapa = ag.Mapa(net, ruta); mapa.archivo_net = cfg["archivo_net"]
    r = ag.correr(".", mapa, cfg, con_agente=CON_AGENTE, gui=True, delay=DELAY_MS,
                  seed=SEMILLA, p_error=P_ERROR, politica=POLITICA)
    print("\nResultado de la corrida:")
    for k, v in r.items():
        print(f"  {k:22}: {v}")
    input("\nListo. Presiona Enter para cerrar...")

## 3. Load the route and build the curvature map

In [ ]:
import importlib, json, shutil, glob, zipfile
import numpy as np, pandas as pd
import ruta_osm as ro, agente_ruta as ag
importlib.reload(ro); importlib.reload(ag)

CARPETA = "fed54_villanueva_apozol"
zips = [z for z in glob.glob("*.zip") if "fed54" in z]
if not zips:
    raise FileNotFoundError("Upload fed54_villanueva_apozol.zip to the Colab file panel (root).")
zipfile.ZipFile(zips[0]).extractall(CARPETA)

def find(name):
    hits = glob.glob(f"{CARPETA}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found inside the zip.")
    return hits[0]

OSM, EDGES, SEGMENTS = find("fed54.osm"), find("ruta_aristas.txt"), find("tramos_elegidos.csv")
NET = f"{CARPETA}/fed54_opp.net.xml"
net = ro.convertir_a_sumo(OSM, NET)
ids = open(EDGES).read().split()
if any(not net.hasEdge(e) for e in ids):
    route = ro.construir_ruta(net, ro.geocodificar("Villanueva, Zacatecas, México"),
                              ro.geocodificar("Apozol, Zacatecas, México"))
else:
    route = [net.getEdge(e) for e in ids]
open(f"{CARPETA}/ruta_aristas_opp.txt", "w").write(" ".join(e.getID() for e in route))

mapa = ag.Mapa(net, route); mapa.archivo_net = NET
curves = ro.detectar_curvas(mapa.s, mapa.kap, radio_max=400)
segments = pd.read_csv(SEGMENTS)
print(f"Route length: {mapa.longitud/1000:.1f} km | curves with R < 400 m: {len(curves)}")
segments

## 4. Experiments (with checkpoint)

In [ ]:
import random, itertools, time

SPEED      = 100                       # km/h
FLOW       = 300                       # veh/h, oncoming traffic
P_ERRORS   = [0.0, 0.05, 0.10, 0.20]   # classification error probability
POLICIES   = ["ultima", "conservadora"]   # last-look / conservative
SEEDS      = list(range(1, 31))        # 30 obstacle placements per segment

CKPT = f"{OUT}/results_final.csv"
done = pd.read_csv(CKPT) if os.path.exists(CKPT) else pd.DataFrame()
key = lambda seg, seed, p, pol, cond: f"{seg}|{seed}|{p}|{pol}|{cond}"
done_keys = set(done.apply(lambda r: key(r.segment, r.seed, r.p_error, r.politica, r.condicion), axis=1)) if len(done) else set()
print(f"Runs already in checkpoint: {len(done_keys)}")

def save(row):
    pd.DataFrame([row]).to_csv(CKPT, mode="a", header=not os.path.exists(CKPT), index=False)

total = len(segments) * len(SEEDS) * (len(P_ERRORS) * len(POLICIES) + 1)
t0, n = time.time(), len(done_keys)
for (k, sg), seed in itertools.product(segments.iterrows(), SEEDS):
    ini, fin = sg.inicio_km * 1000, sg.fin_km * 1000
    obs = ag.obstaculos_aleatorios(mapa, curves, ini, fin, random.Random(seed))
    seg = f"S{k+1} (km {sg.inicio_km:.1f}–{sg.fin_km:.1f})"
    for p, pol in itertools.product(P_ERRORS, POLICIES):
        folder = f"runs/S{k+1}_seed{seed}_p{int(round(p*100))}_{pol}"
        jobs = [("con_agente", True)] + ([("sin_agente", False)] if (p == 0.0 and pol == "conservadora") else [])
        tramo = None
        for cond, with_agent in jobs:
            kk = key(seg, seed, p if with_agent else 0.0, pol if with_agent else "—", cond)
            if kk in done_keys:
                continue
            if tramo is None:
                tramo = ag.preparar_tramo(folder, mapa, ini, fin, SPEED, obs, FLOW)
            r = ag.correr(folder, mapa, tramo, con_agente=with_agent, seed=seed, p_error=p, politica=pol,
                          guardar_pasos=(seed == 1))
            r.update({"segment": seg, "seed": seed}); save(r); done_keys.add(kk); n += 1
            el = time.time() - t0
            print(f"\r{n}/{total} runs | {el/60:.1f} min", end="")
print("\nDone.")
df = pd.read_csv(CKPT)
df.head()

## 5. Summary tables (95 % Wilson confidence intervals)

In [ ]:
from math import sqrt

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    ph = k / n
    den = 1 + z**2 / n
    c = (ph + z**2 / (2 * n)) / den
    h = z * sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / den
    return 100 * ph, 100 * max(0, c - h), 100 * min(1, c + h)

def summarize(g):
    col = wilson(g.colision_obst_denso.sum(), g.obst_densos.sum())
    unn = wilson(g.maniobra_innecesaria.sum(), g.obst_blandos.sum())
    return pd.Series({
        "runs": len(g),
        "dense_obstacles": int(g.obst_densos.sum()),
        "collision_%": col[0], "collision_CI95": f"[{col[1]:.1f}, {col[2]:.1f}]",
        "unnecessary_maneuver_%": unn[0], "unnecessary_CI95": f"[{unn[1]:.1f}, {unn[2]:.1f}]",
        "vehicle_collisions": int(g.colision_vehiculo.sum()),
        "detection_distance_m": g.dist_deteccion_m.mean(),
        "max_lateral_acc_ms2": g.a_lat_max.max(),
        "time_over_threshold_s": g.t_sobre_umbral_s.sum(),
        "mean_wait_s": g.t_espera_s.mean(),
        "mean_travel_time_s": g.tiempo_recorrido_s.mean(),
    })

names = {"con_agente": "Agent", "sin_agente": "No agent", "ultima": "Last-look",
         "conservadora": "Conservative", "—": "—"}
dfe = df.assign(condition=df.condicion.map(names), policy=df.politica.map(names))
table1 = dfe.groupby(["condition", "policy", "p_error"]).apply(summarize).round(2)
table1.to_csv(f"{OUT}/table_summary.csv")
table1

In [ ]:
# Per segment (conservative policy and baseline)
table2 = dfe[dfe.policy.isin(["Conservative", "—"])].groupby(["segment", "condition", "p_error"]).apply(summarize).round(2)
table2.to_csv(f"{OUT}/table_per_segment.csv")
table2

In [ ]:
# LaTeX version of the main table (for Overleaf)
tex = table1.reset_index()
tex["p_error"] = (tex.p_error * 100).round().astype(int)
tex = tex[["condition", "policy", "p_error", "runs", "collision_%", "collision_CI95",
           "unnecessary_maneuver_%", "unnecessary_CI95", "max_lateral_acc_ms2", "mean_travel_time_s"]]
tex.columns = ["Condition", "Policy", "Error (%)", "Runs", "Collision (%)", "95% CI",
               "Unnecessary (%)", "95% CI ", "Max a_lat (m/s2)", "Travel time (s)"]
latex = tex.to_latex(index=False, escape=True, float_format="%.1f",
                     caption="Agent performance under perception uncertainty (95\\% Wilson confidence intervals).",
                     label="tab:uncertainty")
open(f"{OUT}/table_summary.tex", "w").write(latex)
print(latex)

## 6. Figures (300 dpi)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({"font.size": 10})

# --- Figure 1: speed and lateral-acceleration profile on segment 2
c = "runs/S2_seed1_p0_conservadora"
cfg = json.load(open(f"{c}/tramo.json"))
base = pd.read_csv(f"{c}/pasos_base.csv"); agt = pd.read_csv(f"{c}/pasos_agente.csv")
base, agt = base[base.s_m >= cfg["ini_m"] - 1], agt[agt.s_m >= cfg["ini_m"] - 1]   # exclude warm-up zone
km = lambda d: d.s_m / 1000

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.4, 1.4]})
ax[0].plot(km(agt), agt.radio_m.clip(upper=1000), color="#444", lw=.9)
ax[0].set_yscale("log"); ax[0].set_ylim(max(20, 0.8 * agt.radio_m.min()), 1000); ax[0].set_ylabel("Curve radius (m)")
ax[1].plot(km(base), base.v_kmh, color="#999"); ax[1].plot(km(agt), agt.v_kmh, color="#1f77b4")
ax[1].set_ylabel("Speed (km/h)")
ax[2].plot(km(base), base.a_lat, color="#999"); ax[2].plot(km(agt), agt.a_lat, color="#1f77b4")
ax[2].axhline(ag.A_LAT_MAX, color="red", ls="--", lw=1)
ax[2].set_ylabel("Lateral acceleration (m/s²)"); ax[2].set_xlabel("Distance from Villanueva (km)")
for o in cfg["obstaculos"]:
    for a in ax:
        a.axvline(o["s"] / 1000, color="red" if o["densidad"] == "denso" else "green", ls=":", lw=1.2)
for a in ax: a.grid(alpha=.3)
ax[1].legend(handles=[Line2D([], [], color="#999", label="No agent"),
                      Line2D([], [], color="#1f77b4", label="Agent"),
                      Line2D([], [], color="red", ls=":", label="Dense obstacle"),
                      Line2D([], [], color="green", ls=":", label="Soft obstacle")],
             loc="lower left", fontsize=8, ncol=2)
ax[2].legend(handles=[Line2D([], [], color="red", ls="--", label=f"Threshold = {ag.A_LAT_MAX:.0f} m/s²")],
             loc="upper right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_profile.png", dpi=300); plt.show()

In [ ]:
# --- Figure 2: degradation with classification error (95 % Wilson CIs)
ag_df = dfe[dfe.condition == "Agent"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for pol, colr, mk, dx in (("Last-look", "#d62728", "s", -0.3), ("Conservative", "#1f77b4", "o", 0.3)):
    rows_c, rows_u = [], []
    for p, g in ag_df[ag_df.policy == pol].groupby("p_error"):
        rows_c.append((p, *wilson(g.colision_obst_denso.sum(), g.obst_densos.sum())))
        rows_u.append((p, *wilson(g.maniobra_innecesaria.sum(), g.obst_blandos.sum())))
    for a, rows in ((ax[0], rows_c), (ax[1], rows_u)):
        r = np.array(rows)
        a.errorbar(r[:, 0] * 100 + dx, r[:, 1], yerr=[r[:, 1] - r[:, 2], r[:, 3] - r[:, 1]],
                   marker=mk, color=colr, capsize=4, label=pol)
ax[0].set_title("Collisions with dense obstacles"); ax[1].set_title("Unnecessary maneuvers (soft obstacles)")
for a in ax:
    a.set_xticks([0, 5, 10, 20]); a.set_xlabel("Classification error (%)"); a.set_ylabel("Rate (%)")
    a.grid(alpha=.3)
ax[0].legend(title="No agent: 100 %", title_fontsize=8, loc="upper left")
ax[1].legend(loc="upper left")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_degradation.png", dpi=300); plt.show()

In [ ]:
# --- Figure 3: curvature profile of the whole route with the three evaluated segments
R = np.clip(1 / np.maximum(np.abs(mapa.kap), 1e-6), 0, 1000)
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(mapa.s / 1000, R, color="#333", lw=.6)
for _, sg in segments.iterrows():
    ax.axvspan(sg.inicio_km, sg.fin_km, color="#1f77b4", alpha=.15)
for r_, c_ in ((250, "orange"), (150, "red")):
    ax.axhline(r_, ls="--", lw=1, color=c_, label=f"R = {r_} m")
ax.set_yscale("log"); ax.set_ylim(40, 1000)
ax.set_xlabel("Distance from Villanueva (km)"); ax.set_ylabel("Curve radius (m)")
ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=.3)
ax.set_title("Federal Highway 54, Villanueva – Apozol (shaded: evaluated segments)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_route_curvature.png", dpi=300); plt.show()

## 7. Download results and one scenario for local visualization

In [ ]:
SCENARIO = "S2_seed1_p10_conservadora"
dest = f"view_{SCENARIO}"
shutil.rmtree(dest, ignore_errors=True); shutil.copytree(f"runs/{SCENARIO}", dest)
for f in ["ruta_osm.py", "agente_ruta.py", "ver_tramo.py"]:
    shutil.copy(f, dest)
shutil.copy(NET, f"{dest}/fed54_opp.net.xml")
shutil.copy(f"{CARPETA}/ruta_aristas_opp.txt", f"{dest}/ruta_aristas.txt")
cfg = json.load(open(f"{dest}/tramo.json")); cfg["archivo_net"] = "fed54_opp.net.xml"
json.dump(cfg, open(f"{dest}/tramo.json", "w"), indent=2)

!cd {dest} && zip -q -r ../{dest}.zip .
!cd "{OUT}" && zip -q -r /content/final_results.zip results_final.csv table_*.csv table_summary.tex fig_*.png
from google.colab import files
files.download("/content/final_results.zip"); files.download(f"{dest}.zip")